# Gemma-4-E2B — autoLRP attribution

Google's Gemma-4-E2B-it (instruction-tuned, ~2B effective parameters). Same architecture family as Llama and Qwen3 — RMSNorm + RoPE + GQA + GLU MLP. Fused attention is decomposed by autoLRP during the forward.

In [ ]:
import sys
sys.path.insert(0, '..')          # examples/showcase: _common
sys.path.insert(0, '../../..')    # repo root: autoLRP (or `pip install -e .`)
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

import autoLRP as autolrp
from autoLRP import LRPConfig
from _common import show_text_attribution, SHOWCASE_PROMPTS

device = 'cuda' if torch.cuda.is_available() else 'cpu'
MODEL_ID = 'google/gemma-4-E2B-it'
MODEL_NAME = 'Gemma-4'
FILE_STEM = 'gemma'

In [ ]:
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, torch_dtype=torch.float32
).eval().to(device)

class GemmaFromEmb(torch.nn.Module):
    """Embeddings → next-token logits (mean-centered for LRP)."""
    def __init__(self, m):
        super().__init__()
        self.body = m.model
        self.head = m.lm_head
    def forward(self, inputs_embeds):
        h = self.body(inputs_embeds=inputs_embeds).last_hidden_state
        logits = self.head(h)[:, -1, :]
        return logits

wrapper = GemmaFromEmb(model).eval().to(device)

In [ ]:
for pi, text in enumerate(SHOWCASE_PROMPTS):
    ids = tok(text, return_tensors='pt').input_ids.to(device)
    emb = model.model.embed_tokens(ids).detach()

    with torch.no_grad():
        pred = wrapper(emb).argmax(-1).item()
    predicted = tok.decode([pred])

    x = autolrp.tensor(emb)
    out = wrapper(x)
    out[0, pred].lrp(config=LRPConfig.composite())

    tokens = tok.convert_ids_to_tokens(ids[0])
    rel = x.relevance[0].sum(-1).detach().cpu().numpy()
    show_text_attribution(tokens, rel, predicted_token=predicted,
                         prompt_text=f'{MODEL_NAME}: "{text}" \u2192 "{predicted}"')